
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# Streaming Query Lab
### Coupon Sales

Process and append streaming data on transactions using coupons.

##### Objectives
1. Read data stream
2. Filter for transactions with coupons codes
3. Write streaming query results to Delta
4. Monitor streaming query
5. Stop streaming query

##### Classes
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamReader.html" target="_blank">DataStreamReader</a>
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.html" target="_blank">DataStreamWriter</a>
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.html" target="_blank">StreamingQuery</a>

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

In [0]:
%run ./Includes/Classroom-Setup-02L

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.



### 1. Read data stream

Assign the resulting DataFrame to **`df`**:
- Read from Delta files in the source directory specified by **`/Volumes/dbacademy_ecommerce/v01/delta/sales_hist`**
- Set to process 1 file per trigger



In [0]:
df = (
    spark
    .readStream
    .option("maxFilesPerTrigger", 1)
    .format("delta")
    .load('/Volumes/dbacademy_ecommerce/v01/delta/sales_hist'))

print(df.isStreaming)

display(df)

order_id,email,transaction_timestamp,total_item_quantity,purchase_revenue_in_usd,unique_items,items
285712,wbrown@gonzales-miranda.com,1592521889512254,2,1071.0,1,"List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 1071.0, 595.0, 2))"
277921,campbellkatrina@phillips-duarte.com,1592458670364116,1,850.5,1,"List(List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1))"
274566,gregorytorres@meyer.com,1592419954844631,1,535.5,1,"List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))"
257606,ryanolson@brooks.com,1592213705353885,2,1754.0,2,"List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1), List(null, P_FOAM_S, Standard Foam Pillow, 59.0, 59.0, 1))"
257628,kimberly68@mcpherson.net,1592214238807084,1,1045.0,1,"List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))"
289539,levans13@hotmail.com,1592572270401950,2,2016.0,2,"List(List(NEWBED10, M_STAN_K, Standard King Mattress, 1075.5, 1195.0, 1), List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))"
257794,anthonylopez@gmail.com,1592218840420069,1,1045.0,1,"List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))"
257918,wlarson@sanchez.info,1592221370370320,1,1695.0,1,"List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1))"
257798,hernandezjohnny@ball.com,1592218929139881,1,1695.0,1,"List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1))"
258127,laura40@reynolds.com,1592224798871113,1,1195.0,1,"List(List(null, M_STAN_K, Standard King Mattress, 1195.0, 1195.0, 1))"


In [0]:
%skip
df = (spark
      .readStream
      .option("maxFilesPerTrigger", 1)
      .format("delta")
      .load('/Volumes/dbacademy_ecommerce/v01/delta/sales_hist')
     )


**1.1: CHECK YOUR WORK**

In [0]:
# Define the list of required columns
sales_required_columns = [
    'order_id', 'email', 'transaction_timestamp',
    'total_item_quantity', 'purchase_revenue_in_usd',
    'unique_items', 'items'
]

In [0]:
DA.validate_dataframe(df,sales_required_columns)

All validations passed!


### 2. Filter for transactions with coupon codes
- Explode the **`items`** field in **`df`** with the results replacing the existing **`items`** field
- Filter for records where **`items.coupon`** is not null

Assign the resulting DataFrame to **`coupon_sales_df`**.

In [0]:
from pyspark.sql.functions import col, explode

coupon_sales_df = (
    df.withColumn("items", explode(col("items")))
      .filter(col("items.coupon").isNotNull())
)

display(coupon_sales_df)

order_id,email,transaction_timestamp,total_item_quantity,purchase_revenue_in_usd,unique_items,items
285712,wbrown@gonzales-miranda.com,1592521889512254,2,1071.0,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 1071.0, 595.0, 2)"
277921,campbellkatrina@phillips-duarte.com,1592458670364116,1,850.5,1,"List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1)"
274566,gregorytorres@meyer.com,1592419954844631,1,535.5,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1)"
289539,levans13@hotmail.com,1592572270401950,2,2016.0,2,"List(NEWBED10, M_STAN_K, Standard King Mattress, 1075.5, 1195.0, 1)"
289539,levans13@hotmail.com,1592572270401950,2,2016.0,2,"List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1)"
278572,jdixon@steele-kennedy.net,1592474913337711,1,535.5,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1)"
298991,jamie3419@hotmail.com,1592636268357203,1,940.5,1,"List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1)"
275792,meganmiller@gmail.com,1592427703572736,1,535.5,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1)"
281527,randolphjimmy@hotmail.com,1592498653047331,1,535.5,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1)"
288311,hallryan9@gmail.com,1592560621183642,2,1686.6,2,"List(NEWBED10, M_PREM_Q, Premium Queen Mattress, 1615.5, 1795.0, 1)"


In [0]:
%skip
from pyspark.sql.functions import col, explode

coupon_sales_df = (df
                   .withColumn("items", explode(col("items")))
                   .filter(col("items.coupon").isNotNull())
                  )


**2.1: CHECK YOUR WORK**

In [0]:
# Expected schema fields and types

expected_fields = {
    "order_id": "LongType",
    "email": "StringType",
    "transaction_timestamp": "LongType",
    "total_item_quantity": "LongType",
    "purchase_revenue_in_usd": "DoubleType",
    "unique_items": "LongType",
    "items": "StructType"
}

In [0]:
DA.validate_schema(coupon_sales_df.schema,expected_fields)

Schema validation passed!



### 3. Write streaming query results to Delta
- Configure the streaming query to write Delta format files in "append" mode
- Set the query name to "coupon_sales"
- Set a trigger interval of 1 second
- Set the checkpoint location to **`coupons_checkpoint_path`**
- Set the output path to **`coupons_output_path`**

Start the streaming query and assign the resulting handle to **`coupon_sales_query`**.

In [0]:
coupons_checkpoint_path = f"{DA.paths.working_dir}/coupon-sales"
coupons_output_path = f"{DA.paths.working_dir}/coupon-sales/output"

coupon_sales_query = (coupon_sales_df
                      .writeStream
                      .outputMode("append")
                      .format("delta")
                      .queryName("coupon_sales")
                      .trigger(processingTime="1 second")
                      .option("checkpointLocation", coupons_checkpoint_path )
                      .start(coupons_output_path)
)

In [0]:
%skip

coupons_checkpoint_path = f"{DA.paths.working_dir}/coupon-sales"
coupons_output_path = f"{DA.paths.working_dir}/coupon-sales/output"

coupon_sales_query = (coupon_sales_df
                      .writeStream
                      .outputMode("append")
                      .format("delta")
                      .queryName("coupon_sales")
                      .trigger(processingTime="1 second")
                      .option("checkpointLocation", coupons_checkpoint_path)
                      .start(coupons_output_path))


**3.1: CHECK YOUR WORK**

Note: Please wait for stream to get started before validating

In [0]:
DA.validate_coupon_sales_query(coupon_sales_query,coupons_checkpoint_path,coupons_output_path)

Coupon sales query is active.
Found at least one file in /Volumes/dbacademy/ops/labuser13429858_1768515742@vocareum_com/coupon-sales.
Found at least one file in /Volumes/dbacademy/ops/labuser13429858_1768515742@vocareum_com/coupon-sales/output.


### 4. Monitor streaming query
- Get the ID of streaming query and store it in **`queryID`**
- Get the status of streaming query and store it in **`queryStatus`**

In [0]:
query_id = coupon_sales_query.id
query_id

'7278ee78-e3a3-46f4-ade6-54bce2dbb3e4'

In [0]:
%skip
query_id = coupon_sales_query.id
print(query_id)

In [0]:
query_status = coupon_sales_query.status
query_status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [0]:
%skip
query_status = coupon_sales_query.status
print(query_status)

In [0]:
coupon_sales_query.lastProgress

{
    "id": "7278ee78-e3a3-46f4-ade6-54bce2dbb3e4",
    "runId": "9081b9a8-b912-4df6-94d4-ad0a922b742c",
    "name": "coupon_sales",
    "timestamp": "2026-01-15T22:47:21.001Z",
    "batchId": 1,
    "batchDuration": 47,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "latestOffset": 47,
        "triggerExecution": 47
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "DeltaSource[dbfs:/Volumes/dbacademy_ecommerce/v01/delta/sales_hist]",
            "startOffset": {
                "sourceVersion": 1,
                "reservoirId": "ef536d98-b8fe-4c7f-99fe-3dcc7c149068",
                "reservoirVersion": 1,
                "index": -1,
                "isStartingVersion": false
            },
            "endOffset": {
                "sourceVersion": 1,
                "reservoirId": "ef536d98-b8fe-4c7f-99fe-3dcc7c149068",
                "reservoirVersion": 1,
                "i


**4.1: CHECK YOUR WORK**

In [0]:
DA.validate_query_status(query_status)

Test Passed: Valid query status.



### 5. Stop streaming query
- Stop the streaming query

In [0]:
coupon_sales_query.stop()
coupon_sales_query.awaitTermination()

In [0]:
%skip
coupon_sales_query.stop()
coupon_sales_query.awaitTermination()

**5.1: CHECK YOUR WORK**

In [0]:
DA.validate_query_state(coupon_sales_query)

Test Passed: The query is not active.


### 6. Verify the records were written in Delta format

In [0]:
display(spark.read.load(coupons_output_path))

order_id,email,transaction_timestamp,total_item_quantity,purchase_revenue_in_usd,unique_items,items
285712,wbrown@gonzales-miranda.com,1592521889512254,2,1071.0,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 1071.0, 595.0, 2)"
277921,campbellkatrina@phillips-duarte.com,1592458670364116,1,850.5,1,"List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1)"
274566,gregorytorres@meyer.com,1592419954844631,1,535.5,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1)"
289539,levans13@hotmail.com,1592572270401950,2,2016.0,2,"List(NEWBED10, M_STAN_K, Standard King Mattress, 1075.5, 1195.0, 1)"
289539,levans13@hotmail.com,1592572270401950,2,2016.0,2,"List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1)"
278572,jdixon@steele-kennedy.net,1592474913337711,1,535.5,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1)"
298991,jamie3419@hotmail.com,1592636268357203,1,940.5,1,"List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1)"
275792,meganmiller@gmail.com,1592427703572736,1,535.5,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1)"
281527,randolphjimmy@hotmail.com,1592498653047331,1,535.5,1,"List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1)"
288311,hallryan9@gmail.com,1592560621183642,2,1686.6,2,"List(NEWBED10, M_PREM_Q, Premium Queen Mattress, 1615.5, 1795.0, 1)"


In [0]:
%skip
display(spark.read.format("delta").load(coupons_output_path))

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>